### Import libraries

In [1]:
import json
import pandas as pd

### Load the raw OSM JSON export

In [2]:
with open('../data/raw/export.json') as f:
    data = json.load(f)

print("Total elements found:", len(data['elements']))

Total elements found: 24743


### Extract and clean the relevant fields

In [3]:
rows = []

for el in data["elements"]:
    tags = el.get("tags", {})

    # Only keep actual settlements
    if tags.get("place") not in ["city", "town", "village", "hamlet"]:
        continue

    rows.append({
        "id": el.get("id"),
        "name": tags.get("name:en", tags.get("name")),
        "place_type": tags.get("place"),
        "latitude": el.get("lat"),
        "longitude": el.get("lon"),
        "population": tags.get("population")
    })

df = pd.DataFrame(rows)

print("Settlement records:", len(df))
df.head()

Settlement records: 24743


,id,name,place_type,latitude,longitude,population
0,58876734,Landi Kotal,town,34.100527,71.146850,33697
1,66319295,Ghakhar Mandi,town,32.304166,74.146501,NaN
2,81842063,Adiala,town,33.457523,72.995594,NaN
3,81844596,Khasala Khurd,village,33.439479,72.973001,NaN
4,90529210,Kamra,village,33.856090,72.394136,3917


### Check data quality

In [4]:
print("Missing names:", df["name"].isna().sum())
print("Missing coordinates:",
      df[["latitude", "longitude"]].isna().sum().sum())

print("Duplicate name + coordinates:",
      df.duplicated(
          subset=["name", "latitude", "longitude"]
      ).sum())

Missing names: 953
Missing coordinates: 0
Duplicate name + coordinates: 1


### Remove duplicates

In [5]:
df = df.drop_duplicates(
    subset=["name", "latitude", "longitude"]
).copy()

print("Final settlement rows:", len(df))

Final settlement rows: 24742


### Save the cleaned CSV

In [6]:
df.to_csv(
    "../data/processed/osm_settlements_cleaned.csv",
    index=False
)